# Concave Closures from Hard $c$-Transforms

This notebook generates `fig:dual-alternating-c-transform-failure`.  For the bilinear cost
$$
    c(x,y)=-xy,
$$
the $c$-transform is a concave envelope operation on compact one-dimensional domains: $f^{c\bar c}$ is the smallest $c$-concave majorant of $f$, which here is the ordinary concave majorant.  This makes hard alternating best responses very abrupt: one closure removes non-concave oscillations, after which the process has essentially no gradual dynamics.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf,
    box_axes, interp_color,
)

setup_matplotlib()

NAME = "dual-alternating-c-transform-failure"
OUT = figure_dir(NAME)


## Discretized bilinear transform

The grids for $x$ and $y$ are identical.  The source potential $f$ and target potential $g$ are deliberately oscillating and non-concave.  We compute
$$
    f^c(y)=\min_x -xy-f(x),\qquad
    g^{\bar c}(x)=\min_y -xy-g(y),
$$
and then the double transforms $f^{c\bar c}$ and $g^{\bar c c}$.

In [ ]:

x = np.linspace(-1.25, 1.25, 900)
y = x.copy()

f = (
    0.28 - 0.48 * x**2
    + 0.105 * np.sin(9.0 * x + 0.25)
    + 0.055 * np.sin(19.0 * x - 0.5)
    - 0.060 * np.exp(-42.0 * (x - 0.58)**2)
)
f -= f.mean()

# Start the target potential close to the best response to f, then add visible non-concave oscillations.
def c_transform(values, grid_from, grid_to):
    C = -grid_from[:, None] * grid_to[None, :]
    return (C - values[:, None]).min(axis=0)

fc = c_transform(f, x, y)
g = fc + 0.105 * np.sin(8.0 * y - 0.35) - 0.050 * np.sin(18.0 * y + 0.20)
g -= g.mean()

fc = c_transform(f, x, y)
fcc = c_transform(fc, y, x)
gc = c_transform(g, y, x)
gcc = c_transform(gc, x, y)

# Vertical gauge shifts make the three curves in each panel comparable without changing their shape.
gc_vis = gc - np.median(gc - fcc)
fc_vis = fc - np.median(fc - gcc)


## Export source and target panels

Red curves are the original oscillating potentials.  Violet curves are their double-transform closures.  Blue dashed curves are the one-sided best responses generated by the opposite potential.

In [ ]:

def finish_function_axis(ax, grid, curves):
    curves = np.vstack(curves)
    lo = curves.min() - 0.08
    hi = curves.max() + 0.08
    ax.set_xlim(grid.min(), grid.max())
    ax.set_ylim(lo, hi)
    ax.tick_params(labelbottom=False, labelleft=False)
    box_axes(ax)

fig, ax = plt.subplots(figsize=(2.72, 1.92))
ax.plot(x, f, color=RED, lw=1.00, alpha=0.82, zorder=2)
ax.plot(x, gc_vis, color=BLUE, lw=1.05, linestyle=(0, (3.0, 2.0)), alpha=0.92, zorder=3)
ax.plot(x, fcc, color=VIOLET, lw=1.55, zorder=4)
finish_function_axis(ax, x, [f, gc_vis, fcc])
ax.text(x[-1] - 0.30, np.interp(x[-1] - 0.30, x, f), r"$f$", color=RED, fontsize=8)
ax.text(x[-1] - 0.46, np.interp(x[-1] - 0.46, x, gc_vis), r"$g^{\bar c}$", color=BLUE, fontsize=8)
ax.text(x[40], np.interp(x[40], x, fcc) + 0.025, r"$f^{c\bar c}$", color=VIOLET, fontsize=8)
save_pdf(fig, OUT / "source-concave-closure.pdf", pad_inches=0.060)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.72, 1.92))
ax.plot(y, g, color=RED, lw=1.00, alpha=0.82, zorder=2)
ax.plot(y, fc_vis, color=BLUE, lw=1.05, linestyle=(0, (3.0, 2.0)), alpha=0.92, zorder=3)
ax.plot(y, gcc, color=VIOLET, lw=1.55, zorder=4)
finish_function_axis(ax, y, [g, fc_vis, gcc])
ax.text(y[-1] - 0.32, np.interp(y[-1] - 0.32, y, g), r"$g$", color=RED, fontsize=8)
ax.text(y[55], np.interp(y[55], y, fc_vis) + 0.035, r"$f^c$", color=BLUE, fontsize=8)
ax.text(y[-1] - 0.62, np.interp(y[-1] - 0.62, y, gcc) + 0.025, r"$g^{\bar c c}$", color=VIOLET, fontsize=8)
save_pdf(fig, OUT / "target-concave-closure.pdf", pad_inches=0.060)
plt.close(fig)
